# Лабораторная работа 8. Законы масштабирования и Mixture of Experts (студенческая версия)

**Курс:** Методы обработки естественного языка · **Блок 3:** предобучение, оптимизация, масштабирование

**Цель.** Получить собственные законы масштабирования в миниатюре, найти compute-optimal соотношение размера модели и числа токенов и исследовать разреженные MoE-модели.

**Что нужно сдать:**
1. Серию обученных моделей разного размера при нескольких бюджетах токенов.
2. Аппроксимацию \(L(N, D) = E + A/N^\alpha + B/D^\beta\) с bootstrap-интервалами для \(\alpha, \beta\).
3. IsoFLOP-анализ и оценку compute-optimal отношения \(D/N\) в сравнении с Chinchilla.
4. MoE-слой с top-2 роутингом и балансирующим loss; сравнение с плотной моделью при равном числе активных параметров.
5. Анализ коллапса роутинга и специализации экспертов.

**Теория.** При ограничении бюджета \(C = 6ND\) минимизация \(L(N, D) = E + A N^{-\alpha} + B D^{-\beta}\) даёт
\[
N_{opt}(C) = G\left(\frac{C}{6}\right)^{\frac{\beta}{\alpha+\beta}},\qquad
D_{opt}(C) = G^{-1}\left(\frac{C}{6}\right)^{\frac{\alpha}{\alpha+\beta}},\qquad
G = \left(\frac{\alpha A}{\beta B}\right)^{\frac{1}{\alpha+\beta}} .
\]
В работе Hoffmann et al. (2022) \(\alpha \approx 0.34\), \(\beta \approx 0.28\), что даёт около 20 токенов на параметр.

**Балансирующий loss (Switch Transformer).** Для \(E\) экспертов, доли токенов \(f_i\), направленных к эксперту \(i\), и средней вероятности роутера \(P_i\):
\[
\mathcal{L}_{aux} = E\sum_{i=1}^{E} f_i P_i ,
\]
минимум (равный 1) достигается при равномерной загрузке.

In [ ]:
# ==== Общая настройка (одинакова во всех работах) ====
import os, math, time, json, random, pathlib, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

FAST_DEV_RUN = True          # True — сокращённый прогон (CPU, минуты); False — полный эксперимент
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = pathlib.Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)
DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

def download(url, name):
    path = DATA / name
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path

def load_shakespeare():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    return download(url, "shakespeare.txt").read_text(encoding="utf-8")

def load_russian():
    """Русский корпус: «Война и мир» (Л. Н. Толстой), т. 1, из открытого репозитория.
    При недоступности сети используется встроенный фрагмент."""
    urls = ["https://raw.githubusercontent.com/mmcdermott/nlp-war-and-peace/master/data/war_and_peace_ru.txt",
            "https://raw.githubusercontent.com/nevmenandr/word2vec-russian-novels/master/books_before/WarAndPeace.txt"]
    for i, u in enumerate(urls):
        try:
            t = download(u, f"ru_{i}.txt").read_text(encoding="utf-8", errors="ignore")
            if len(t) > 100_000: return t
        except Exception:
            pass
    base = ("Все счастливые семьи похожи друг на друга, каждая несчастливая семья несчастлива по-своему. "
            "Всё смешалось в доме Облонских. Жена узнала, что муж был в связи с бывшею в их доме француженкою-гувернанткой, "
            "и объявила мужу, что не может жить с ним в одном доме. ")
    return base * 400

print(f"torch {torch.__version__}, device={DEVICE}, FAST_DEV_RUN={FAST_DEV_RUN}")

In [ ]:
# ==== Эталонный mini-GPT (решение ЛР 6), используется как готовый модуль ====
from dataclasses import dataclass, asdict

class CharTokenizer:
    def __init__(self, text):
        self.chars = sorted(set(text)); self.stoi = {c: i for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars)
    def encode(self, s): return [self.stoi[c] for c in s if c in self.stoi]
    def decode(self, ids): return "".join(self.chars[i] for i in ids)

@dataclass
class GPTConfig:
    vocab_size: int = 65
    block_size: int = 128
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    dropout: float = 0.0

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def rope_cache(T, d_head, base=10000.0, device="cpu", scale=1.0):
    inv = 1.0 / (base ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    t = torch.arange(T, device=device).float() / scale
    ang = torch.outer(t, inv)                        # (T, d_head/2)
    return ang.cos(), ang.sin()

def apply_rope(x, cos, sin):                         # x: (B, H, T, d_head)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.flatten(-2)

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head, self.d_head = cfg.n_head, cfg.n_embd // cfg.n_head
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.d_head).transpose(1, 2) for t in (q, k, v))
        q, k = apply_rope(q, cos[:T], sin[:T]), apply_rope(k, cos[:T], sin[:T])
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class SwiGLU(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.w1, self.w3 = nn.Linear(d, hidden, bias=False), nn.Linear(d, hidden, bias=False)
        self.w2 = nn.Linear(hidden, d, bias=False)
    def forward(self, x): return self.w2(F.silu(self.w1(x)) * self.w3(x))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1, self.attn = RMSNorm(cfg.n_embd), CausalSelfAttention(cfg)
        self.n2, self.mlp = RMSNorm(cfg.n_embd), SwiGLU(cfg.n_embd, int(8 * cfg.n_embd / 3))
    def forward(self, x, cos, sin):
        x = x + self.attn(self.n1(x), cos, sin)
        return x + self.mlp(self.n2(x))

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm = RMSNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.head.weight = self.wte.weight                    # weight tying
        cos, sin = rope_cache(4 * cfg.block_size, cfg.n_embd // cfg.n_head)
        self.register_buffer("cos", cos, persistent=False); self.register_buffer("sin", sin, persistent=False)
        self.apply(self._init)
        for n, p in self.named_parameters():                  # масштабирование выходных проекций по глубине
            if n.endswith("proj.weight") or n.endswith("w2.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * cfg.n_layer))
    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)): nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None):
        x = self.wte(idx)
        for b in self.blocks: x = b(x, self.cos, self.sin)
        logits = self.head(self.norm(x))
        loss = None if targets is None else F.cross_entropy(logits.flatten(0, 1), targets.flatten())
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            probs = F.softmax(logits[:, -1] / temperature, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

def n_params(model):
    return sum(p.numel() for p in model.parameters())

def get_batch(data, block_size, batch_size, device=DEVICE):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix]); y = torch.stack([data[i + 1:i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model, data, block_size, batch_size=32, iters=20):
    model.eval(); losses = [model(*get_batch(data, block_size, batch_size))[1].item() for _ in range(iters)]
    model.train(); return float(np.mean(losses))

def train_lm(model, train_data, val_data, steps, lr=3e-3, batch_size=32, warmup=50, log_every=100):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warmup) * 0.5 * (1 + math.cos(math.pi * min(s, steps) / steps)))
    hist = []
    for s in range(steps):
        x, y = get_batch(train_data, model.cfg.block_size, batch_size)
        _, loss = model(x, y); opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
        if s % log_every == 0 or s == steps - 1:
            hist.append((s, loss.item(), estimate_loss(model, val_data, model.cfg.block_size)))
    return hist

def get_reference_gpt(text, cfg=None, steps=None, name=None):
    """Загружает или обучает эталонную символьную модель на тексте."""
    tok = CharTokenizer(text)
    data = torch.tensor(tok.encode(text), dtype=torch.long)
    n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]
    cfg = cfg or (GPTConfig(tok.vocab_size, 64, 2, 4, 64) if FAST_DEV_RUN else GPTConfig(tok.vocab_size, 256, 6, 6, 384))
    name = name or f"ref_gpt_L{cfg.n_layer}_d{cfg.n_embd}_T{cfg.block_size}.pt"
    path = ARTIFACTS / name
    model = GPT(cfg).to(DEVICE)
    if path.exists():
        model.load_state_dict(torch.load(path, map_location=DEVICE))
    else:
        train_lm(model, train_data, val_data, steps or (300 if FAST_DEV_RUN else 5000))
        torch.save(model.state_dict(), path)
    return model, tok, train_data, val_data

In [ ]:
text = load_shakespeare(); tok = CharTokenizer(text)
data = torch.tensor(tok.encode(text)); ntr = int(0.9 * len(data)); train_data, val_data = data[:ntr], data[ntr:]
T_CTX, BATCH = 64, 32
TOKENS_PER_STEP = T_CTX * BATCH

## 1. Серия моделей

Используйте эталонный GPT с фиксированной глубиной и варьируйте ширину. Число параметров \(N\) считайте **без эмбеддингов** (как у Kaplan et al.). Для каждой модели оцените val loss при нескольких бюджетах токенов \(D\).

Корректная процедура требует отдельного запуска с полным косинусным расписанием для каждой пары \((N, D)\). В `FAST_DEV_RUN` используется упрощение: один запуск с постоянным LR и оценкой в промежуточных точках. Оно завышает loss для малых \(D\) (не было фазы спада LR), и это надо учесть при интерпретации.

In [ ]:
WIDTHS = [32, 48, 64, 96, 128] if FAST_DEV_RUN else [32, 48, 64, 96, 128, 192, 256, 384]
STEP_BUDGETS = [50, 100, 200, 400] if FAST_DEV_RUN else [250, 500, 1000, 2000, 4000]
SEPARATE_RUNS = not FAST_DEV_RUN
N_LAYER = 2 if FAST_DEV_RUN else 4

def non_embedding_params(model):
    return sum(p.numel() for n, p in model.named_parameters() if not n.startswith(("wte", "head")))

def run_scaling(width, budgets, lr=3e-3, separate=False):
    """Возвращает (N, список (D, val_loss))."""
    cfg = GPTConfig(tok.vocab_size, T_CTX, N_LAYER, max(1, width // 16), width)
    # TODO: ваш код (при separate — отдельный train_lm на каждый бюджет; иначе один запуск с постоянным LR и оценками)
    raise NotImplementedError

t0 = time.time(); points = []
for w in WIDTHS:
    N, res = run_scaling(w, STEP_BUDGETS, separate=SEPARATE_RUNS)
    for D, L in res: points.append((N, D, L))
    print(f"d={w:4d}  N={N:>9,}  " + "  ".join(f"D={D:,}: {L:.3f}" for D, L in res))
points = np.array(points); print(f"{time.time() - t0:.0f} c")
np.save(ARTIFACTS / "lab08_points.npy", points)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for N in np.unique(points[:, 0]):
    p = points[points[:, 0] == N]; ax[0].plot(p[:, 1], p[:, 2], "o-", label=f"N={int(N):,}")
for D in np.unique(points[:, 1]):
    p = points[points[:, 1] == D]; ax[1].plot(p[:, 0], p[:, 2], "o-", label=f"D={int(D):,}")
for a, xl in zip(ax, ["D, токенов", "N, параметров без эмбеддингов"]):
    a.set_xscale("log"); a.set_yscale("log"); a.set_xlabel(xl); a.set_ylabel("val loss"); a.legend(fontsize=7)
plt.tight_layout(); plt.show()

## 2. Аппроксимация \(L(N, D)\) и bootstrap

Подберите \((E, A, \alpha, B, \beta)\) методом наименьших квадратов в пространстве \(\log L\) (`scipy.optimize.least_squares`, ограничения \(E, A, B > 0\), \(0 < \alpha, \beta < 2\)), начиная с нескольких начальных точек. Затем выполните bootstrap: 200 раз перевыберите точки с возвращением и постройте 90%-интервалы для \(\alpha\) и \(\beta\).

In [ ]:
from scipy.optimize import least_squares

def law(params, N, D):
    E, A, alpha, B, beta = params
    return E + A / N ** alpha + B / D ** beta

def fit_law(pts, n_starts=20, seed=0):
    # TODO: ваш код (параметризуйте E, A, B через логарифмы; остатки log(law) - log(L); несколько стартов)
    raise NotImplementedError

params, cost = fit_law(points)
E, A, alpha, B, beta = params
print(f"E={E:.3f}, A={A:.3g}, α={alpha:.3f}, B={B:.3g}, β={beta:.3f}, cost={cost:.2e}")
pred = law(params, points[:, 0], points[:, 1])
print("средняя относительная ошибка:", np.mean(np.abs(pred - points[:, 2]) / points[:, 2]).round(4))

In [ ]:
# TODO: ваш код (bootstrap 200 (в FAST 50) повторов, 90% интервалы для α и β)

## 3. IsoFLOP и compute-optimal соотношение

1. Для нескольких бюджетов \(C\) постройте \(L(N, C/6N)\) как функцию \(N\) (IsoFLOP-кривые) по подобранному закону и найдите минимум.
2. Сравните численный минимум с аналитической формулой \(N_{opt}(C)\) из введения.
3. Посчитайте оптимальное число токенов на параметр \(D_{opt}/N_{opt}\) и сравните с ~20 у Chinchilla. Объясните расхождение: символьная токенизация, маленькие модели, вклад эмбеддингов, упрощённая процедура.

In [ ]:
# TODO: ваш код (IsoFLOP-кривые, аналитический и численный оптимум, D/N)

## 4. Mixture of Experts

Реализуйте MoE-FFN с \(E\) экспертами (каждый — SwiGLU со скрытым размером \(h_e\)) и top-\(k\) роутингом:
1. логиты роутера \(z = xW_r\), вероятности \(p = \mathrm{softmax}(z)\);
2. выбор top-\(k\) экспертов, веса — перенормированные вероятности выбранных экспертов;
3. балансирующий loss \(\mathcal{L}_{aux} = E\sum_i f_i P_i\), где \(f_i\) — доля назначений токенов эксперту \(i\) (с учётом всех \(k\) выборов, нормированная на \(k\)), \(P_i\) — средняя вероятность.

Плотная модель с FFN-скрытым размером \(h\) и MoE с \(E = 4\), \(k = 2\), \(h_e = h/2\) имеют **одинаковое число активных параметров** на токен, но у MoE вдвое больше общих параметров FFN.

In [ ]:
class Expert(nn.Module):
    def __init__(self, d, h):
        super().__init__(); self.w1, self.w3, self.w2 = nn.Linear(d, h, bias=False), nn.Linear(d, h, bias=False), nn.Linear(h, d, bias=False)
    def forward(self, x): return self.w2(F.silu(self.w1(x)) * self.w3(x))

class MoE(nn.Module):
    def __init__(self, d, n_experts=4, hidden=None, k=2):
        super().__init__()
        self.E, self.k = n_experts, k
        self.router = nn.Linear(d, n_experts, bias=False)
        self.experts = nn.ModuleList([Expert(d, hidden or int(8 * d / 3) // 2) for _ in range(n_experts)])
        self.aux, self.last_assign = torch.tensor(0.0), None
    def forward(self, x):
        B, T, C = x.shape; xf = x.reshape(-1, C)
        # TODO: ваш код (top-k по вероятностям, перенормировка весов, цикл по экспертам с index_add_, aux loss)
        raise NotImplementedError
        return out.view(B, T, C)

def make_model(kind, width=128):
    cfg = GPTConfig(tok.vocab_size, T_CTX, N_LAYER, 4, width)
    set_seed(); m = GPT(cfg)
    if kind.startswith("moe"):
        for b in m.blocks: b.mlp = MoE(width)
    elif kind == "dense-small":
        for b in m.blocks: b.mlp = SwiGLU(width, int(8 * width / 3) // 2 * 2)   # та же активная ширина, что top-2 × h/2
    return m.to(DEVICE)

def train_moe(model, steps, aux_coef=0.01, lr=3e-3):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1); hist = []
    for s in range(steps):
        x, y = get_batch(train_data, T_CTX, BATCH); loss = model(x, y)[1]
        aux = sum(b.mlp.aux for b in model.blocks if isinstance(b.mlp, MoE)) if aux_coef else 0.0
        opt.zero_grad(); (loss + aux_coef * aux).backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        if s % 25 == 0 or s == steps - 1:
            loads = [torch.bincount(b.mlp.last_assign.flatten(), minlength=4).float() / b.mlp.last_assign.numel()
                     for b in model.blocks if isinstance(b.mlp, MoE)]
            hist.append((s, loss.item(), [l.tolist() for l in loads]))
    return hist

MOE_STEPS = 300 if FAST_DEV_RUN else 3000
runs = {}
for name, kind, coef in [("dense (активные = MoE)", "dense-small", 0), ("MoE top-2, aux=0.01", "moe", 0.01), ("MoE top-2, без aux", "moe", 0.0)]:
    m = make_model(kind); t0 = time.time(); h = train_moe(m, MOE_STEPS, aux_coef=coef)
    total = sum(p.numel() for p in m.parameters())
    runs[name] = (m, h); print(f"{name:25s} params={total:,}  val={estimate_loss(m, val_data, T_CTX, iters=20):.3f}  ({time.time() - t0:.0f} c)")

### Коллапс роутинга и специализация

1. Постройте динамику загрузки экспертов (доля top-1 назначений) первого слоя с балансирующим loss и без него; посчитайте энтропию загрузки.
2. Для модели с aux loss определите, какие типы символов (строчные буквы, заглавные, пробел и перевод строки, пунктуация) преимущественно направляются к каждому эксперту.

In [ ]:
# TODO: ваш код (графики загрузки экспертов и таблица специализации)

## Вопросы

1. Выведите \(N_{opt}(C)\) из условия Лагранжа для задачи \(\min L(N,D)\) при \(6ND = C\).
2. Почему Kaplan et al. и Hoffmann et al. получили разные compute-optimal соотношения? Какой методологический фактор оказался главным?
3. Почему в практике (LLaMA и др.) малые модели обучают далеко за пределами compute-optimal числа токенов?
4. Почему без балансирующего loss роутинг коллапсирует? Опишите петлю положительной обратной связи.
5. Как сравнивать MoE и плотные модели: по активным параметрам, полным параметрам, FLOPs или памяти? Что выбирать для обучения и для инференса?